# Qwen3.8-27B AI Director — Kaggle Deployment

**7-Stage AETHER Integration Notebook**

Stages: 1 Hardware → 2 Model load → 3 Inference → 4 Structured JSON → 5 AETHER compat → 6 FastAPI+Ngrok → 7 Benchmark

## Stage 1 — Hardware Verification

In [ ]:
import subprocess, sys, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError('nvidia-smi failed — no GPU detected.')

gpu_count = torch.cuda.device_count()
print(f'PyTorch sees {gpu_count} GPU(s)\n')
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    vram = p.total_memory / 1024**3
    print(f'  GPU {i}: {p.name}  VRAM: {vram:.1f} GB')
print(f'\nCUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

if gpu_count < 2:
    raise RuntimeError(
        f'Need 2x T4 but only {gpu_count} GPU(s) found. '
        'Go to Settings → Accelerator → GPU T4 x2 and re-run.'
    )
print('\n✅ Stage 1 PASSED')


## Stage 2 — Install pinned dependencies

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
    'vllm',  # Unpin vLLM so it can install alongside the latest transformers
    'fastapi',
    'uvicorn[standard]',
    'pyngrok',
    'pydantic',
    'openai',
    'git+https://github.com/huggingface/transformers.git', # Bleeding-edge Qwen3 support
    'starlette>=0.49.1,<1.0.0',
    'protobuf>=5.26.1,<7.0.0',
])
print('[STAGE 2] Dependencies installed successfully.', flush=True)



## Stage 2 — Configuration

Set `NGROK_AUTHTOKEN` and `DIRECTOR_API_KEY` via **Kaggle Secrets** (Add-ons → Secrets).

In [ ]:
import os

DIRECTOR_MODEL   = 'cyankiwi/Qwen3.8-27B-AWQ-INT4'
DIRECTOR_TP_SIZE = 2
DIRECTOR_MAX_LEN = 8192
DIRECTOR_EFFORT  = 'low'
VLLM_PORT        = 8000
API_PORT         = 8001

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    NGROK_AUTHTOKEN  = _s.get_secret('NGROK_AUTHTOKEN')
    DIRECTOR_API_KEY = _s.get_secret('DIRECTOR_API_KEY')
except Exception:
    NGROK_AUTHTOKEN  = os.environ.get('NGROK_AUTHTOKEN', '')
    DIRECTOR_API_KEY = os.environ.get('DIRECTOR_API_KEY', 'test-key-change-me')

os.environ['DIRECTOR_MODEL']   = DIRECTOR_MODEL
os.environ['DIRECTOR_API_KEY'] = DIRECTOR_API_KEY
os.environ['VLLM_PORT']        = str(VLLM_PORT)

print('Model  :', DIRECTOR_MODEL)
print('TP     :', DIRECTOR_TP_SIZE)
print('MaxLen :', DIRECTOR_MAX_LEN)
print('Effort :', DIRECTOR_EFFORT)
print('APIKey :', DIRECTOR_API_KEY[:4] + '****')
print('Ngrok  :', 'configured' if NGROK_AUTHTOKEN else 'NOT SET')


## Stage 2 — Write Director Python module

Files are written with `pathlib` to avoid any quoting ambiguity.

In [ ]:
import os, sys
os.makedirs('director', exist_ok=True)
print('[STAGE 2c] Working dir:', os.getcwd(), flush=True)
print('[STAGE 2c] director/ folder ready', flush=True)


In [ ]:
from pathlib import Path

SCHEMA_SRC = 'from __future__ import annotations\nfrom typing import List, Optional, Literal\nfrom pydantic import BaseModel, Field\n\n\nclass ProjectInfo(BaseModel):\n    title: str\n    summary: str\n    tone: str\n    visual_style: str\n    target_audience: str\n\n\nclass StockSearch(BaseModel):\n    queries: List[str] = Field(description="3-5 concrete stock-search queries")\n    negative_terms: List[str] = Field(default_factory=list)\n\n\nclass TextOverlay(BaseModel):\n    enabled: bool\n    text: Optional[str] = None\n    purpose: Optional[str] = None\n    emphasis_level: Optional[Literal["low", "medium", "high", "hero"]] = None\n    animation: Optional[str] = None\n\n\nclass EditorialGraphic(BaseModel):\n    enabled: bool\n    type: Optional[Literal[\n        "chart", "diagram", "map", "timeline",\n        "infographic", "quote_card", "split_screen",\n        "intentional_minimal", "other"\n    ]] = None\n    description: Optional[str] = None\n\n\nclass Visual(BaseModel):\n    type: Literal["stock_video", "stock_image", "text_graphic", "editorial_graphic"]\n    subject: str\n    action: str\n    context: str\n    composition: str\n    mood: str\n\n\nclass Shot(BaseModel):\n    shot_id: str\n    start: float\n    end: float\n    narration: str\n    visual: Visual\n    stock_search: StockSearch\n    importance: Literal["filler", "context", "key_beat", "hero"]\n    text_overlay: TextOverlay\n    editorial_graphic: EditorialGraphic\n\n\nclass Scene(BaseModel):\n    scene_id: str\n    purpose: str\n    start: float\n    end: float\n    shots: List[Shot]\n\n\nclass Storyboard(BaseModel):\n    schema_version: str = "1.0"\n    project: ProjectInfo\n    scenes: List[Scene]\n\n\ndef get_json_schema() -> dict:\n    return Storyboard.model_json_schema()\n'

Path('director/__init__.py').write_text('# Qwen3.8-27B Director package\n', encoding='utf-8')
Path('director/schema.py').write_text(SCHEMA_SRC, encoding='utf-8')
print('wrote schema.py')
print('  wrote director/schema.py', flush=True)


In [ ]:
from pathlib import Path

PROMPTS_SRC = r'''DIRECTOR_SYSTEM_PROMPT = (
    "You are the central AI Intelligence Brain for AETHER, a professional video-production application. "
    "Your job is to assist the user with scriptwriting, story research, video structure, and storyboard generation. "
    "The application uses stock footage (Pixabay/Pexels) and editorial graphics, NOT AI-generated video like LTX. "
    "\n\n"
    "RULES:\n"
    "1. STOCK FOOTAGE VISUALS: When planning visuals, describe concrete searchable concepts. "
    "   For 'inflation erodes purchasing power' use 'expensive grocery shopping' or 'rising food prices'.\n"
    "2. NO FABRICATION: Do not invent nonexistent stock assets or fabricate historical events.\n"
    "3. EDITORIAL GRAPHICS: You can use text overlays for punchy editorial text like '$10 BILLION'.\n"
    "4. JSON OUTPUT (for /director only): If you are asked to generate a storyboard, you MUST return strictly valid JSON matching the schema.\n"
)
'''

Path('director/prompts.py').write_text(PROMPTS_SRC, encoding='utf-8')
print('wrote prompts.py')
print('  wrote director/prompts.py', flush=True)


In [ ]:
from pathlib import Path

CACHE_SRC = 'import hashlib, json, os\n\nCACHE_DIR = "/tmp/director_cache"\n\n\ndef init_cache():\n    os.makedirs(CACHE_DIR, exist_ok=True)\n\n\ndef get_cache_key(script, style, model_id, schema_ver, effort):\n    raw = f"{script}|{style}|{model_id}|{schema_ver}|{effort}"\n    return hashlib.sha256(raw.encode()).hexdigest()\n\n\ndef get_cached_result(cache_key):\n    path = os.path.join(CACHE_DIR, f"{cache_key}.json")\n    if os.path.exists(path):\n        with open(path, encoding="utf-8") as f:\n            return json.load(f)\n    return None\n\n\ndef set_cached_result(cache_key, data):\n    os.makedirs(CACHE_DIR, exist_ok=True)\n    path = os.path.join(CACHE_DIR, f"{cache_key}.json")\n    with open(path, "w", encoding="utf-8") as f:\n        json.dump(data, f)\n'

Path('director/cache.py').write_text(CACHE_SRC, encoding='utf-8')
print('wrote cache.py')
print('  wrote director/cache.py', flush=True)


In [ ]:
from pathlib import Path

INFERENCE_SRC = r'''import json, re, time
from openai import OpenAI
from typing import List, Dict, Any
from .schema import get_json_schema
from .prompts import DIRECTOR_SYSTEM_PROMPT

def _strip_thinking(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

class DirectorInference:
    def __init__(self, model_id: str, port: int = 8000):
        self.model_id = model_id
        self.client = OpenAI(
            base_url=f"http://localhost:{port}/v1",
            api_key="sk-no-key-required",
            timeout=1200,
        )

    def chat(self, messages: List[Dict[str, str]], **kwargs):
        # Prepend the system prompt if not present
        if not any(m.get("role") == "system" for m in messages):
            messages.insert(0, {"role": "system", "content": DIRECTOR_SYSTEM_PROMPT})
        
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
            **kwargs
        )
        # For streaming
        if kwargs.get('stream'):
            return response
        
        return response.choices[0].message.content or ""

    def generate_storyboard(self, script: str, style: str = "documentary", reasoning_effort: str = "low"):
        messages = [
            {"role": "system", "content": DIRECTOR_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Visual style: {style}\n\n"
                    f"Script:\n{script}\n\n"
                    "Return ONLY the JSON storyboard  no markdown, no commentary."
                ),
            },
        ]
        extra_body = {"reasoning_effort": reasoning_effort} if reasoning_effort else {}
        t0 = time.time()
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
            temperature=0.0,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "storyboard",
                    "schema": get_json_schema(),
                    "strict": True,
                },
            },
            extra_body=extra_body or None,
        )
        latency = time.time() - t0
        raw = response.choices[0].message.content or ""
        cleaned = _strip_thinking(raw)
        try:
            return json.loads(cleaned), latency
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON after {latency:.1f}s. Error: {e}. Raw[:500]: {raw[:500]}")
'''

Path('director/inference.py').write_text(INFERENCE_SRC, encoding='utf-8')
print('wrote inference.py')
print('  wrote director/inference.py', flush=True)


In [ ]:
from pathlib import Path

API_SRC = r'''import os, time, json
from fastapi import FastAPI, Depends, HTTPException, Request
from fastapi.responses import StreamingResponse
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel
from typing import Optional, List, Dict, Any
from .inference import DirectorInference
from .cache import init_cache, get_cache_key, get_cached_result, set_cached_result

API_KEY  = os.environ.get("DIRECTOR_API_KEY",  "test-key-change-me")
MODEL_ID = os.environ.get("DIRECTOR_MODEL",    "cyankiwi/Qwen3.8-27B-AWQ-INT4")
VLLM_PORT = int(os.environ.get("VLLM_PORT",   "8000"))

app      = FastAPI(title="Qwen3.8-27B AETHER Brain API")
security = HTTPBearer()
_engine  = None

def get_engine():
    global _engine
    if _engine is None:
        _engine = DirectorInference(model_id=MODEL_ID, port=VLLM_PORT)
    return _engine

def check_token(creds: HTTPAuthorizationCredentials = Depends(security)):
    if creds.credentials != API_KEY:
        raise HTTPException(401, "Invalid API key")
    return creds.credentials

@app.on_event("startup")
async def _startup():
    init_cache()

class ChatRequest(BaseModel):
    messages: List[Dict[str, str]]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 1024
    stream: Optional[bool] = False

class GenerateRequest(BaseModel):
    prompt: str
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 1024
    response_format: Optional[Dict[str, Any]] = None

class DirectorRequest(BaseModel):
    script: str
    title: Optional[str] = "Untitled"
    style: Optional[str] = "documentary"
    language: Optional[str] = "en"
    reasoning_effort: Optional[str] = "low"

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.get("/model")
def model_info():
    return {"model": MODEL_ID}

@app.post("/chat")
async def chat_endpoint(req: ChatRequest, _: str = Depends(check_token)):
    try:
        engine = get_engine()
        if req.stream:
            # Proxy the generator directly
            res = engine.chat(req.messages, temperature=req.temperature, max_tokens=req.max_tokens, stream=True)
            async def stream_generator():
                for chunk in res:
                    yield f"data: {{json.dumps({{'choices': [{{'delta': {{'content': chunk.choices[0].delta.content or ''}}}}]}})}}\n\n"
                yield "data: [DONE]\n\n"
            return StreamingResponse(stream_generator(), media_type="text/event-stream")
        else:
            text = engine.chat(req.messages, temperature=req.temperature, max_tokens=req.max_tokens, stream=False)
            return {"choices": [{"message": {"content": text}}]}
    except Exception as e:
        raise HTTPException(500, str(e))

@app.post("/generate")
def generate_endpoint(req: GenerateRequest, _: str = Depends(check_token)):
    try:
        engine = get_engine()
        kwargs = {"temperature": req.temperature, "max_tokens": req.max_tokens, "stream": False}
        if req.response_format:
            kwargs["response_format"] = req.response_format
        text = engine.chat([{"role": "user", "content": req.prompt}], **kwargs)
        return {"content": text}
    except Exception as e:
        raise HTTPException(500, str(e))

@app.post("/director")
def direct(req: DirectorRequest, _: str = Depends(check_token)):
    cache_key = get_cache_key(req.script, req.style, MODEL_ID, "1.0", req.reasoning_effort)
    cached = get_cached_result(cache_key)
    if cached:
        return {"success": True, "cached": True, "storyboard": cached}
    try:
        sb, latency = get_engine().generate_storyboard(
            script=req.script,
            style=req.style,
            reasoning_effort=req.reasoning_effort,
        )
        set_cached_result(cache_key, sb)
        return {
            "success": True, "cached": False,
            "latency_sec": round(latency, 2), "storyboard": sb,
        }
    except Exception as e:
        raise HTTPException(500, str(e))
'''

Path('director/api.py').write_text(API_SRC, encoding='utf-8')
print('wrote api.py')
print('  wrote director/api.py', flush=True)


In [ ]:
import py_compile, glob, sys

ok = True
for fp in sorted(glob.glob('director/*.py')):
    try:
        py_compile.compile(fp, doraise=True)
        print(f'  OK  {fp}')
    except py_compile.PyCompileError as e:
        print(f'  ERR {fp}: {e}')
        ok = False
if not ok:
    raise RuntimeError('Syntax errors — fix before continuing')
print('\n✅ All module files pass syntax check')


## Stage 2 — Boot vLLM (15 min on first run)

In [ ]:
import subprocess, sys, time, threading
import requests
import torch

print('=' * 70, flush=True)
print('[STAGE 2d] STARTING vLLM MODEL SERVER', flush=True)
print('  Model  :', DIRECTOR_MODEL, flush=True)
print('  TP     :', DIRECTOR_TP_SIZE, flush=True)
print('  MaxLen :', DIRECTOR_MAX_LEN, flush=True)
print('  NOTE: Model download + TP init takes 10-20 minutes.', flush=True)
print('        You will see a heartbeat every 60 seconds.', flush=True)
print('        vLLM log lines appear in cyan below.', flush=True)
print('=' * 70, flush=True)

import os
custom_env = os.environ.copy()
custom_env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
custom_env['VLLM_USE_FLASHINFER'] = '0' # Disable flashinfer completely to avoid any JIT compilation issues

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model',                DIRECTOR_MODEL,
    '--tensor-parallel-size', str(DIRECTOR_TP_SIZE),
    '--max-model-len',        str(DIRECTOR_MAX_LEN),
    '--trust-remote-code',
    '--enforce-eager',        # Disable CUDA graphs to save 1-2GB of VRAM
    '--port',                 str(VLLM_PORT),
]

vllm_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=custom_env,

)

# Background thread: continuously drain and print vLLM log lines
_stop_reader = threading.Event()
def _log_reader():
    for line in iter(vllm_proc.stdout.readline, ''):
        if line:
            print('  [vLLM]', line, end='', flush=True)
        if _stop_reader.is_set():
            break

_reader_thread = threading.Thread(target=_log_reader, daemon=True)
_reader_thread.start()

# Main thread: health-check loop with 60-second heartbeat
t_start         = time.time()
last_beat       = time.time()
BEAT_INTERVAL   = 60
deadline        = t_start + 1200  # 20 min max
booted          = False

while time.time() < deadline:
    time.sleep(5)

    # 60-second heartbeat
    now = time.time()
    if now - last_beat >= BEAT_INTERVAL:
        elapsed   = int(now - t_start)
        remaining = int(deadline - now)
        print(f'  [HEARTBEAT] {elapsed//60:02d}m{elapsed%60:02d}s elapsed — '
              f'{remaining//60}m{remaining%60:02d}s before timeout — '
              f'still waiting for vLLM...', flush=True)
        last_beat = now

    # Did vLLM crash?
    if vllm_proc.poll() is not None:
        _stop_reader.set()
        raise RuntimeError(
            'vLLM process exited before becoming healthy. '
            'Scroll up and read the [vLLM] lines for the error.'
        )

    # Health check
    try:
        r = requests.get(f'http://localhost:{VLLM_PORT}/v1/models', timeout=3)
        if r.status_code == 200:
            booted = True
            break
    except Exception:
        pass

_stop_reader.set()

if not booted:
    raise RuntimeError('vLLM timed out after 20 minutes.')

elapsed = int(time.time() - t_start)
print(f'\n[STAGE 2d] vLLM HEALTHY after {elapsed//60}m{elapsed%60:02d}s', flush=True)
for i in range(torch.cuda.device_count()):
    a = torch.cuda.memory_allocated(i) / 1024**3
    t = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f'  GPU {i}: {a:.1f}/{t:.1f} GB VRAM used', flush=True)
print('[STAGE 2d] PASSED', flush=True)


## Stage 3 — Real Inference Test

In [ ]:
from openai import OpenAI
import time, threading, sys

print("[STAGE 3] Sending first inference request...", flush=True)
print("          (Triton C++ JIT compilation takes 10-15 minutes in the background)", flush=True)
print("          (You will see a heartbeat every 30 seconds)", flush=True)

client = OpenAI(
    base_url=f'http://localhost:{VLLM_PORT}/v1',
    api_key='sk-no-key',
    timeout=3600, max_retries=0,
)

t0 = time.time()
resp_container = {}
exc_container = {}

def _run_inference():
    try:
        resp = client.chat.completions.create(
            model=DIRECTOR_MODEL,
            messages=[{'role': 'user', 'content': 'Say exactly: INFERENCE OK'}],
            max_tokens=20,
            temperature=0.0,
        )
        resp_container['data'] = resp
    except Exception as e:
        exc_container['error'] = e

t = threading.Thread(target=_run_inference)
t.start()

while t.is_alive():
    t.join(timeout=30.0)
    if t.is_alive():
        elapsed = int(time.time() - t0)
        print(f"  [STAGE 3 HEARTBEAT] {elapsed//60:02d}m{elapsed%60:02d}s elapsed — still compiling/inferencing...", flush=True)

if 'error' in exc_container:
    raise exc_container['error']

lat  = time.time() - t0
text = resp_container['data'].choices[0].message.content
print(f'\nResponse : {text}')
print(f'Latency  : {lat:.2f}s')
if not text:
    raise RuntimeError('Empty response — Stage 3 FAILED')
print('\n✅ Stage 3 PASSED')



## Stage 4 — Structured Director JSON + Pydantic Validation

In [ ]:
from director.inference import DirectorInference
from director.schema import Storyboard
import json, time

engine = DirectorInference(model_id=DIRECTOR_MODEL, port=VLLM_PORT)

TEST_SCRIPT = (
    'Inflation quietly reduces what your paycheck can buy over time. '
    'As prices rise, the same salary buys fewer groceries, less fuel, '
    'and fewer everyday essentials. Central banks may respond by raising '
    'interest rates, which affects mortgages, investments, and consumer spending.'
)

results = {}
for effort in ['off', 'low', 'medium']:
    print(f'--- effort={effort} ---')
    try:
        sb_raw, lat = engine.generate_storyboard(
            script=TEST_SCRIPT, style='finance', reasoning_effort=effort)
        sb  = Storyboard(**sb_raw)
        sc  = len(sb.scenes)
        sh  = sum(len(s.shots) for s in sb.scenes)
        print(f'  Latency={lat:.1f}s  Scenes={sc}  Shots={sh}  ✅')
        results[effort] = {'latency': lat, 'scenes': sc, 'shots': sh, 'data': sb_raw}
    except Exception as e:
        print(f'  ❌ {e}')
        results[effort] = {'error': str(e)}

best = min(
    (k for k, v in results.items() if 'data' in v),
    key=lambda k: results[k]['latency'],
    default=None,
)
if best is None:
    raise RuntimeError('All effort configs failed — Stage 4 FAILED')
DIRECTOR_EFFORT  = best
director_result  = results[best]['data']
print(f'\n✅ Stage 4 PASSED — best effort={DIRECTOR_EFFORT}')
print(json.dumps(director_result, indent=2)[:1500], '...')


## Stage 5 — AETHER Compatibility Check

In [ ]:
from director.schema import Storyboard

sb = Storyboard(**director_result)
issues = []

for scene in sb.scenes:
    for shot in scene.shots:
        if not shot.stock_search.queries:
            issues.append(f'{shot.shot_id}: no stock queries')
        for q in shot.stock_search.queries:
            if len(q.split()) < 2:
                issues.append(f'{shot.shot_id}: too-short query: {q!r}')
        valid = {'stock_video','stock_image','text_graphic','editorial_graphic'}
        if shot.visual.type not in valid:
            issues.append(f'{shot.shot_id}: unknown visual.type={shot.visual.type!r}')
        if shot.end <= shot.start:
            issues.append(f'{shot.shot_id}: end <= start')

print('--- AETHER Compatibility Report ---')
for scene in sb.scenes:
    for shot in scene.shots:
        qs = ', '.join(shot.stock_search.queries[:2])
        print(f'  {shot.shot_id}  {shot.visual.type:15}  {qs}')

if issues:
    print('\nIssues:')
    for i in issues:
        print(' ', i)
    raise RuntimeError(f'Stage 5: {len(issues)} compatibility issue(s)')
print('\n✅ Stage 5 PASSED — output is AETHER-compatible')


## Stage 6 — FastAPI + Ngrok External API

In [ ]:
import subprocess, sys, time
import requests as _req

api_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'director.api:app',
     '--host', '0.0.0.0', '--port', str(API_PORT)],
    stdout=open('/tmp/api.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

try:
    r = _req.get(f'http://localhost:{API_PORT}/health', timeout=10)
    print('FastAPI health:', r.json())
except Exception as e:
    print(open('/tmp/api.log').read()[-2000:])
    raise RuntimeError(f'FastAPI failed: {e}')

PUBLIC_URL = None
if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    PUBLIC_URL = ngrok.connect(API_PORT).public_url
    print(f'\n🌐 Public URL: {PUBLIC_URL}')
    print(f'curl -X POST {PUBLIC_URL}/director '
          f'-H "Authorization: Bearer {DIRECTOR_API_KEY}" '
          f'-H "Content-Type: application/json" '
          "-d '{\"script\": \"Test script\"}' ")
else:
    print('Ngrok not configured — localhost only')

# Internal endpoint smoke test
smoke = _req.post(
    f'http://localhost:{API_PORT}/director',
    headers={'Authorization': f'Bearer {DIRECTOR_API_KEY}'},
    json={'script': 'A brief test.', 'style': 'documentary'},
    timeout=1200,
)
print('Endpoint status:', smoke.status_code)
if smoke.status_code != 200:
    print(smoke.text)
    raise RuntimeError('Stage 6 endpoint test FAILED')
print('\n✅ Stage 6 PASSED')


## Stage 7 — 12-Domain Benchmark

In [ ]:
import time, json
from director.inference import DirectorInference
from director.schema import Storyboard

DOMAINS = [
    ('medicine',     'A new study reveals that sleeping fewer than six hours a night significantly increases the risk of heart disease.'),
    ('cooking',      'The secret to a perfect risotto is patience. Add warm stock one ladle at a time, stirring constantly until the starch releases.'),
    ('technology',   'Large language models are trained on billions of tokens of text. They predict the next word based on patterns learned from human writing.'),
    ('science',      'DNA carries the genetic instructions for life. Every cell contains the same sequence of three billion base pairs.'),
    ('history',      'In 1944, Allied forces launched the largest amphibious invasion in history on the beaches of Normandy, turning the tide of World War II.'),
    ('education',    'The Socratic method encourages students to question assumptions rather than passively receive information from a teacher.'),
    ('fitness',      'High-intensity interval training burns more calories in twenty minutes than a sixty-minute steady-state jog.'),
    ('business',     'Startups that focus on customer problems rather than product features are four times more likely to achieve product-market fit.'),
    ('psychology',   'Cognitive dissonance occurs when a person holds two conflicting beliefs and feels discomfort until one is resolved.'),
    ('geography',    'The Amazon rainforest produces twenty percent of the world\'s oxygen and is home to ten percent of all species on Earth.'),
    ('storytelling', 'Every great story follows a simple arc: a character wants something, obstacles appear, and struggle reveals who they truly are.'),
    ('finance',      'Inflation quietly reduces what your paycheck can buy. Central banks raise interest rates to slow spending and cool price growth.'),
]

engine = DirectorInference(model_id=DIRECTOR_MODEL, port=VLLM_PORT)
rows = []

print(f'{"Domain":<15} {"Status":<6} {"Latency":>8} {"Scenes":>6} {"Shots":>6} {"Queries":>8}')
print('-' * 55)

for domain, script in DOMAINS:
    try:
        sb_raw, lat = engine.generate_storyboard(
            script=script, style=domain, reasoning_effort=DIRECTOR_EFFORT)
        sb = Storyboard(**sb_raw)
        sc = len(sb.scenes)
        sh = sum(len(s.shots) for s in sb.scenes)
        qs = sum(len(sh.stock_search.queries)
                 for s in sb.scenes for sh in s.shots)
        print(f'{domain:<15} {"OK":<6} {lat:>7.1f}s {sc:>6} {sh:>6} {qs:>8}')
        rows.append({'domain': domain, 'ok': True, 'latency': lat})
    except Exception as e:
        print(f'{domain:<15} {"FAIL":<6} {str(e)[:35]}')
        rows.append({'domain': domain, 'ok': False})

passed = sum(1 for r in rows if r['ok'])
print(f'\nResult: {passed}/{len(DOMAINS)} domains passed')
if passed == len(DOMAINS):
    print('\n🎉 ALL STAGES COMPLETE — Director is production-ready')
